# Avaliação — Comparação de experiências

Fluxo:

```text
results_pp_X  →  pós-processamento (00_common)  →  métricas  →  tabela comparativa
```

Compara **Original**, **PP1**–**PP5**, com e sem `pos_process()`.

## EVALUATION CONFIGURATION

In [ ]:
# ==========================================================
# EVALUATION CONFIGURATION
# ==========================================================

# Corrigir orientação das previsões (como em Pos_Metrics.ipynb)
APLICAR_CORRECAO_ORIENTACAO = True

# Experiências a avaliar: (nome_tabela, pasta_resultados em 02_dataset/)
EXPERIMENTOS_AVALIAR = [
    ("Original", "results_original"),
    ("PP1", "results_pp_1"),
    ("PP2", "results_pp_2"),
    ("PP3", "results_pp_3"),
    ("PP4", "results_pp_4"),
    ("PP5", "results_pp_5"),
]

## Biblioteca comum de pós-processamento

In [ ]:
%run ../03_postprocessing/postprocessing_common.ipynb

## Validação global (paths, datasets, emparelhamentos)

In [ ]:
import pandas as pd
from IPython.display import display

print("=" * 60)
print("VALIDAÇÃO DE PATHS E DATASETS")
print("=" * 60)

avisos = validar_paths_projeto()
if avisos:
    for aviso in avisos:
        print(f"[AVISO] {aviso}")
else:
    print("Pastas principais encontradas.")

print("\nEmparelhamento previsão ↔ label (por ID base):")
relatorio_pairing = validar_todos_emparelhamentos()
for chave, erros in relatorio_pairing.items():
    if erros:
        print(f"  {chave}: {len(erros)} erro(s) — ex.: {erros[0]}")
    else:
        print(f"  {chave}: OK")

## Avaliar cada experiência (com / sem pós-processamento)

In [ ]:
def avaliar_experiencia(nome_experiencia: str, pasta_resultados_nome: str) -> list:
    """
    Avalia todas as previsões numa pasta de resultados.
    Devolve duas linhas de métricas: sem e com pós-processamento.
    """
    pasta_resultados = DATA_DIR / pasta_resultados_nome
    previsoes = listar_previsoes(pasta_resultados)

    if len(previsoes) == 0:
        return [
            {
                "Experiência": nome_experiencia,
                "Pós": "Não",
                "N": 0,
                "Dice": np.nan,
                "Accuracy": np.nan,
                "Precision": np.nan,
                "Recall": np.nan,
                "Pasta": pasta_resultados_nome,
            },
            {
                "Experiência": nome_experiencia,
                "Pós": "Sim",
                "N": 0,
                "Dice": np.nan,
                "Accuracy": np.nan,
                "Precision": np.nan,
                "Recall": np.nan,
                "Pasta": pasta_resultados_nome,
            },
        ]

    metricas_sem_pos = []
    metricas_com_pos = []

    for caminho_previsao in previsoes:
        caminho_label = resolver_caminho_label(caminho_previsao.name)

        predicted = carregar_mascara_png(caminho_previsao)
        gt = carregar_mascara_png(caminho_label)

        if APLICAR_CORRECAO_ORIENTACAO:
            predicted = corrigir_orientacao_previsao(predicted)

        # Métricas sem pós-processamento (Pos_Metrics — primeiro bloco)
        dice, ac, pr, re = calculate_metrics(predicted, gt)
        metricas_sem_pos.append((dice, ac, pr, re))

        # Pós-processamento + métricas (Pos_Metrics — segundo bloco)
        predicted_pos_proc = pos_process(predicted)
        dice, ac, pr, re = calculate_metrics(predicted_pos_proc, gt)
        metricas_com_pos.append((dice, ac, pr, re))

    media_sem = calcular_metricas_medias(metricas_sem_pos)
    media_com = calcular_metricas_medias(metricas_com_pos)

    return [
        {
            "Experiência": nome_experiencia,
            "Pós": "Não",
            "N": len(previsoes),
            "Dice": media_sem["dice"],
            "Accuracy": media_sem["accuracy"],
            "Precision": media_sem["precision"],
            "Recall": media_sem["recall"],
            "Pasta": pasta_resultados_nome,
        },
        {
            "Experiência": nome_experiencia,
            "Pós": "Sim",
            "N": len(previsoes),
            "Dice": media_com["dice"],
            "Accuracy": media_com["accuracy"],
            "Precision": media_com["precision"],
            "Recall": media_com["recall"],
            "Pasta": pasta_resultados_nome,
        },
    ]


linhas_tabela = []
for nome_exp, pasta_res in EXPERIMENTOS_AVALIAR:
    linhas_tabela.extend(avaliar_experiencia(nome_exp, pasta_res))

tabela_comparativa = pd.DataFrame(linhas_tabela)
colunas_ordem = [
    "Experiência",
    "Pós",
    "N",
    "Dice",
    "Accuracy",
    "Precision",
    "Recall",
    "Pasta",
]
tabela_comparativa = tabela_comparativa[colunas_ordem]

## Tabela comparativa final

In [ ]:
pd.set_option("display.max_rows", 20)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")

display(tabela_comparativa)

caminho_csv = PROJECT_ROOT / "04_pipeline_results" / "tabela_avaliacao_experiencias.csv"
caminho_csv.parent.mkdir(parents=True, exist_ok=True)
tabela_comparativa.to_csv(caminho_csv, index=False)
print(f"\nTabela guardada em: {caminho_csv}")

## Visualização resumo (Dice)

In [ ]:
import matplotlib.pyplot as plt

if len(tabela_comparativa) > 0 and tabela_comparativa["Dice"].notna().any():
    fig, ax = plt.subplots(figsize=(10, 5))
    for aplicar_pos, grupo in tabela_comparativa.groupby("Pós"):
        ax.plot(
            grupo["Experiência"],
            grupo["Dice"],
            marker="o",
            label=f"Pós-processamento: {aplicar_pos}",
        )
    ax.set_ylabel("Dice (média)")
    ax.set_xlabel("Experiência")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()